# 1A pilot v2 — Diacritic noise vs. accuracy, 4–8B models, item-level analysis

> **Tóm tắt (VI):** Bản v2 của pilot 1A. So với v1: (1) đổi sang *batch calibration* vì cách cũ làm hỏng Qwen3; (2) thêm chẩn đoán để biết vì sao một mô hình chỉ đoán ngẫu nhiên (SeaLLMs); (3) dùng mô hình 4–8B thay cho 1.5B; (4) thêm điều kiện `mixed` (teen code + bỏ dấu 50%); (5) phân tích theo từng câu: câu bị "lật" từ đúng sang sai là do tokenizer hay do mô hình ít gặp văn bản không dấu?

**Changes from v1** (v1 results: `MyDrive/llm-research/1A-pilot/`)

| v1 problem | v2 fix |
|---|---|
| Content-free calibration ("", "N/A") collapsed Qwen3 to one class (raw 0.67 → 0.34) | **Batch calibration** ([Zhou et al. 2024](https://arxiv.org/abs/2309.17249)): the prior is the mean label distribution over the test items of each condition |
| SeaLLMs at chance even on clean text. Cause: it answers correctly but in its own format ("Nhãn: tiêu cực", "Phân loại … là tiêu cực"), so first-token label scoring missed it | **Prefill the answer with "Nhãn:"** for every model, plus **diagnostics**: 6 free greedy answers and the probability mass on the 3 labels |
| 3 of 6 models at chance → floor effects made the correlation look strong | **4–8B models**, a **sanity gate** (clean accuracy ≥ 0.45 to enter the analysis), and a **normalised drop** |
| Teen code changed almost nothing | Larger rule table + **`mixed`** = teen code, then strip 50% |
| ~30 model × condition points: too few for "tokenizer vs exposure" | **Item-level logistic regression**: does a flip follow extra tokens / broken syllables, or extra NLL? |

**Runtime:** L4 GPU. About 8–10 min per 7B model (download + run), so roughly 1–1.5 h and 3–5 CU in total. Each finished model is saved, so a disconnect only loses the model in progress.

⚠️ When you are done: `Runtime → Disconnect and delete runtime`.

## 0. Setup

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U transformers accelerate

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/llm-research/1A-pilot-v2"
else:
    OUT_DIR = "outputs/1A-pilot-v2"
os.makedirs(f"{OUT_DIR}/preds", exist_ok=True)
print("Saving results to", OUT_DIR)

In [ ]:
# Only needed for gated models. In Colab: 🔑 Secrets panel → add HF_TOKEN,
# and accept each model's licence on its Hugging Face page first.
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret — gated models will be skipped.")

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory / 1e9:.1f} GB")
    # T4 (compute capability 7.5) has no real bfloat16, so it gets float16
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    print("⚠️ No GPU. Runtime → Change runtime type → L4 GPU")
    DTYPE = torch.float32
print("dtype:", DTYPE)

## 1. Config

The models are chosen so that **some share a tokenizer but not their training data**: Qwen2.5-7B, Qwen3-8B and SeaLLMs-v3-7B (continued training on Southeast-Asian text) all use the Qwen tokenizer. If their drops differ, the tokenizer alone cannot be the explanation. The others bring different tokenizers: VinaLLaMA (Llama-2 vocabulary extended with Vietnamese), Phi-3.5 (Llama-2 vocabulary, 32k), bloomz (250k multilingual), Mistral (32k). With an `HF_TOKEN`, Vistral-7B (Mistral continued on Vietnamese) gives a second "same base, more Vietnamese" pair next to Mistral-7B.

A model that fails to load is skipped and listed at the end, not a crash.

In [ ]:
MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen3-8B",
    "SeaLLMs/SeaLLMs-v3-7B-Chat",          # same tokenizer as Qwen, more SEA data
    "vilm/vinallama-7b-chat",              # Llama-2 + Vietnamese vocabulary
    "microsoft/Phi-3.5-mini-instruct",     # 3.8B, Llama-2 vocabulary
    "bigscience/bloomz-7b1",               # multilingual, no chat template
    "mistralai/Mistral-7B-Instruct-v0.3",  # Mistral vocabulary, 32k
]
GATED_MODELS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Viet-Mistral/Vistral-7B-Chat",        # Mistral + Vietnamese, compare with Mistral-7B
    "CohereLabs/aya-expanse-8b",
]
if HF_TOKEN:
    MODELS += GATED_MODELS

CONDITIONS = ["clean", "partial25", "partial50", "partial75", "strip", "teencode", "mixed"]
N_PER_CLASS = 100   # balanced sample: 100 negative, 100 neutral, 100 positive
SEED = 42
BATCH_SIZE = 8
MIN_CLEAN_ACC = 0.45   # sanity gate: below this a model cannot "drop", so it is left out of the analysis
DELETE_WEIGHTS_AFTER_USE = IN_COLAB   # 7B weights are ~15 GB each; keeps Colab's disk from filling up

## 2. Noise generator

- **Partial removal** strips an exact share of the syllables *that carry diacritics*.
- **Teen code** is a rule table. ✏️ Check it yourself and extend it — it is still small.
- **Mixed** = teen code, then strip 50% of the remaining diacritics: closer to how people actually type in a hurry.
- Noise is applied only to the student's sentence, never to the instruction or the labels.

In [ ]:
import random, re, unicodedata

def strip_diacritics(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s)

def partial_strip(text, rate, rng):
    words = text.split(" ")
    candidates = [i for i, w in enumerate(words) if strip_diacritics(w) != w]
    chosen = set(rng.sample(candidates, round(rate * len(candidates))))
    return " ".join(strip_diacritics(w) if i in chosen else w for i, w in enumerate(words))

TEENCODE = {
    "giáo viên": "gv", "giảng viên": "gv", "sinh viên": "sv", "bài tập": "bt", "học sinh": "hs",
    "không": "ko", "được": "dc", "gì": "j", "với": "vs", "mình": "mk", "rồi": "r",
    "vậy": "v", "bạn": "bn", "quá": "wa", "nhiều": "nhìu", "người": "ng", "trước": "trc",
    "cũng": "cx", "thôi": "thui", "biết": "bít", "yêu": "iu", "giờ": "h",
    "nhưng": "nhg", "lắm": "lém", "thích": "thik", "những": "nhữg", "hiểu": "hỉu",
}
_teen_re = re.compile(
    r"(?<!\w)(" + "|".join(sorted(map(re.escape, TEENCODE), key=len, reverse=True)) + r")(?!\w)",
    re.IGNORECASE,
)

def teencode(text):
    return _teen_re.sub(lambda m: TEENCODE[m.group(0).lower()], text)

def make_noisy(text, cond, item_id):
    rng = random.Random(f"{SEED}-{item_id}-{cond}")  # same noise every run
    if cond == "clean":
        return text
    if cond == "strip":
        return strip_diacritics(text)
    if cond.startswith("partial"):
        return partial_strip(text, int(cond[len("partial"):]) / 100, rng)
    if cond == "teencode":
        return teencode(text)
    if cond == "mixed":
        return partial_strip(teencode(text), 0.5, rng)
    raise ValueError(cond)

# sanity checks
assert strip_diacritics("Tôi muốn đặt vé máy bay") == "Toi muon dat ve may bay"
assert teencode("giáo viên không nói gì") == "gv ko nói j"
s = "thầy dạy nhiệt tình nhưng bài tập không hiểu lắm"
for c in CONDITIONS:
    print(f"{c:10} {make_noisy(s, c, 0)}")

## 3. Data: UIT-VSFC (students' feedback sentiment), test split

In [ ]:
import pandas as pd

URL = ("https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/"
       "refs%2Fconvert%2Fparquet/default/test/0000.parquet")
raw = pd.read_parquet(URL)
print(len(raw), "test rows; class counts:", raw.sentiment.value_counts().sort_index().to_dict())

sample = raw.groupby("sentiment").sample(n=N_PER_CLASS, random_state=SEED).reset_index(drop=True)
sample["item_id"] = range(len(sample))

rows = [{"item_id": r.item_id, "label": r.sentiment, "cond": c, "clean": r.sentence,
         "text": make_noisy(r.sentence, c, r.item_id)}
        for r in sample.itertuples() for c in CONDITIONS]
data = pd.DataFrame(rows)
data.to_csv(f"{OUT_DIR}/noisy_items.csv", index=False)

changed = (data.text != data.clean).groupby(data.cond).mean()
print("share of items changed by each condition:\n", changed.round(2).to_string())
data[data.item_id == 0][["cond", "text"]]

## 4. Tokenizer statistics, per item (CPU, a few minutes)

Kept **per item** this time, so section 7 can ask whether the items that break are the ones the tokenizer fragments most.

- **fertility** = tokens ÷ syllables of the *clean* sentence (same denominator for every condition).
- **broken_rate** = share of syllables split into more than one token.
- **fragment_rate** = share of tokens that are not a whole character on their own.

In [ ]:
from transformers import AutoTokenizer
from tqdm.auto import tqdm

def syllable_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text) if any(ch.isalpha() for ch in m.group())]

def item_tok_stats(tok, text, clean):
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    spans = syllable_spans(text)
    return {
        "n_tok": len(ids),
        "n_syl": len(spans),
        "clean_syl": len(syllable_spans(clean)),
        "n_broken": sum(sum(1 for s, e in offs if e > a and s < b and e > s) > 1 for a, b in spans),
        "n_frag": sum("�" in tok.decode([i]) for i in ids),
    }

tok_rows, tok_failed = [], {}
for name in tqdm(MODELS):
    try:
        tok = AutoTokenizer.from_pretrained(name, token=HF_TOKEN)
    except Exception as e:
        tok_failed[name] = repr(e)[:200]
        continue
    for r in data.itertuples():
        tok_rows.append({"model": name, "item_id": r.item_id, "cond": r.cond, **item_tok_stats(tok, r.text, r.clean)})
tok_items = pd.DataFrame(tok_rows)
tok_items.to_csv(f"{OUT_DIR}/tokenizer_items.csv", index=False)
for name, err in tok_failed.items():
    print("✗ tokenizer failed, model skipped:", name, err)
MODELS = [m for m in MODELS if m not in tok_failed]

g = tok_items.groupby(["model", "cond"]).sum(numeric_only=True)
tok_stats = pd.DataFrame({
    "fertility": g.n_tok / g.clean_syl,
    "broken_rate": g.n_broken / g.n_syl,
    "fragment_rate": g.n_frag / g.n_tok,
}).reset_index()
tok_stats.to_csv(f"{OUT_DIR}/tokenizer_stats.csv", index=False)
tok_stats.pivot(index="model", columns="cond", values="fertility")[CONDITIONS].round(2)

## 5. Model evaluation (GPU)

For each item we score the three labels ("tiêu cực", "trung tính", "tích cực") by their log-probability after the prompt, summing both casings. No generation, so no answer parsing.

**New in v2: the answer is prefilled with "Nhãn:"** and the labels are scored right after it. In v1, SeaLLMs answered correctly but in its own words ("Nhãn: tiêu cực", "Phân loại cảm xúc của câu nhận xét là tiêu cực."), so scoring the label as the first answer token measured its answer format, not its judgement.

**Two predictions are saved:**
- `pred_raw`: the label with the highest log-probability.
- `pred`: **batch-calibrated**. Subtract the model's average label preference over all items of the same condition, then take the highest. This removes "always says Tiêu cực" bias without the fragile content-free inputs of v1.

**Diagnostics, printed for every model:**
- `label_mass`: the share of probability the model puts on our three labels at all. If it is tiny (say < 5%), the model wants to answer in another form, and label scoring measures little. This is the first thing to check for a model at chance (v1: SeaLLMs).
- 6 greedy answers to clean items, saved to `diagnostics.csv`, so you can see what the model actually says.

We also measure **NLL per clean syllable** of the sentence itself (how "surprised" the model is by the noisy text).

In [ ]:
import gc, time, shutil
import torch.nn.functional as F
from transformers import AutoModelForCausalLM
from huggingface_hub import constants as hf_constants

LABELS = ["tiêu cực", "trung tính", "tích cực"]  # 0 negative, 1 neutral, 2 positive
INSTRUCTION = ("Phân loại cảm xúc của câu nhận xét sau của sinh viên. "
               "Trả lời bằng đúng một nhãn: tiêu cực, trung tính, hoặc tích cực.\n\nCâu: {text}")
NLL_CONTEXT = "Nhận xét của sinh viên:"
# v1 bug: SeaLLMs answers "Nhãn: tiêu cực" or "Phân loại ... là tiêu cực", so scoring the label as the
# very first answer token measured its answer *format*. Prefilling "Nhãn:" puts every model in the same slot.
ANSWER_PREFILL = "Nhãn:"

def prompt_text(tok, text, prefill=True):
    msg = INSTRUCTION.format(text=text)
    if tok.chat_template:
        # enable_thinking=False only matters for Qwen3; other templates ignore it
        s = tok.apply_chat_template([{"role": "user", "content": msg}], tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
        return s + (ANSWER_PREFILL if prefill else ""), False
    return f"{msg}\n{ANSWER_PREFILL}", True

def prompt_ids(tok, text, prefill=True):
    s, add_special = prompt_text(tok, text, prefill)
    return tok(s, add_special_tokens=add_special)["input_ids"]

def label_ids(tok):
    # per label: [lowercase, Capitalised] token ids, after "Nhãn:" so with a leading space
    return [[tok(" " + v, add_special_tokens=False)["input_ids"] for v in (l, l.capitalize())] for l in LABELS]

def nll_ids(tok, text):
    # score only the sentence, after a short clean context (Qwen has no BOS token to start from)
    s = f"{NLL_CONTEXT} {text}"
    enc = tok(s, add_special_tokens=True, return_offsets_mapping=True)
    first = next(i for i, (a, b) in enumerate(enc["offset_mapping"]) if b > a and a >= len(NLL_CONTEXT))
    return enc["input_ids"], first

@torch.no_grad()
def score(model, pad_id, seqs, starts):
    # seqs[i][starts[i]:] is the scored part; returns (sum log-prob, n scored tokens) per sequence
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), pad_id)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    ids, mask = ids.to(model.device), mask.to(model.device)
    logits = model(input_ids=ids, attention_mask=mask).logits
    pick = torch.zeros_like(mask, dtype=torch.bool)  # pick[i, t]: logits at t predict token t+1
    for i, (s, st) in enumerate(zip(seqs, starts)):
        pick[i, st - 1:len(s) - 1] = True
    lp = F.log_softmax(logits[pick].float(), dim=-1)
    tgt = ids[:, 1:][pick[:, :-1]]
    tok_lp = lp.gather(1, tgt[:, None]).squeeze(1)
    owner = pick.nonzero()[:, 0]
    sums = torch.zeros(len(seqs), device=lp.device).index_add_(0, owner, tok_lp)
    counts = pick.sum(1)
    return sums.cpu().tolist(), counts.cpu().tolist()

@torch.no_grad()
def greedy_answers(model, tok, texts, max_new_tokens=20):
    out = []
    for t in texts:
        ids = torch.tensor([prompt_ids(tok, t, prefill=False)], device=model.device)  # free answer, no prefill
        gen = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
        out.append(tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True).strip())
    return out

def batched(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]

def evaluate(name):
    tok = AutoTokenizer.from_pretrained(name, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, token=HF_TOKEN).to(DEVICE).eval()
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)
    lab = label_ids(tok)

    # diagnostics: what does the model say when it may answer freely?
    probe = sample.groupby("sentiment").head(2)
    diag = pd.DataFrame({"model": name, "item_id": probe.item_id, "label": probe.sentiment,
                         "sentence": probe.sentence, "answer": greedy_answers(model, tok, list(probe.sentence))})

    # label scoring: 6 sequences per item (3 labels × 2 casings)
    seqs, starts = [], []
    for text in data.text:
        p = prompt_ids(tok, text)
        for variants in lab:
            for v in variants:
                seqs.append(p + v)
                starts.append(len(p))
    lps = []
    for chunk in tqdm(list(batched(list(zip(seqs, starts)), BATCH_SIZE)), desc="labels", leave=False):
        lps += score(model, pad_id, *zip(*chunk))[0]
    lp = torch.tensor(lps).view(len(data), 3, 2).logsumexp(-1)

    # sentence NLL: 1 sequence per item
    nll_seqs = [nll_ids(tok, t) for t in data.text]
    nll, ntok = [], []
    for chunk in tqdm(list(batched(nll_seqs, BATCH_SIZE)), desc="nll", leave=False):
        s, c = score(model, pad_id, *zip(*chunk))
        nll += [-x for x in s]
        ntok += c

    out = data[["item_id", "label", "cond"]].copy()
    out["model"] = name
    for k in range(3):
        out[f"lp{k}"] = lp[:, k].tolist()
    out["label_mass"] = lp.logsumexp(-1).exp().tolist()
    out["pred_raw"] = lp.argmax(1).tolist()
    # batch calibration: subtract the mean label distribution of the same condition
    logp = lp.log_softmax(-1)
    cal = torch.empty_like(logp)
    for c in CONDITIONS:
        idx = torch.tensor((data.cond == c).values)
        cal[idx] = logp[idx] - logp[idx].exp().mean(0).log()
    out["pred"] = cal.argmax(1).tolist()
    out["nll"] = nll
    out["nll_tokens"] = ntok
    out["clean_syl"] = [len(syllable_spans(c)) for c in data.clean]

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if DELETE_WEIGHTS_AFTER_USE:
        shutil.rmtree(os.path.join(hf_constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
    return out, diag

def pred_path(name):
    return f"{OUT_DIR}/preds/{name.replace('/', '__')}.csv"

failed = {}
for name in MODELS:
    if os.path.exists(pred_path(name)):
        print("✓ already done:", name)
        continue
    t0 = time.time()
    try:
        res, diag = evaluate(name)
    except Exception as e:  # e.g. out of memory, or a licence not accepted
        failed[name] = repr(e)[:300]
        print("✗ failed:", name, failed[name])
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        continue
    res.to_csv(pred_path(name), index=False)
    diag_path = f"{OUT_DIR}/diagnostics.csv"
    diag.to_csv(diag_path, index=False, mode="a", header=not os.path.exists(diag_path))
    acc = (res.pred == res.label).groupby(res.cond).mean()
    raw_acc = (res.pred_raw == res.label).groupby(res.cond).mean()
    mass = res[res.cond == "clean"].label_mass.median()
    print(f"✓ {name} ({time.time() - t0:.0f}s)  clean={acc['clean']:.2f} (raw {raw_acc['clean']:.2f})  "
          f"strip={acc['strip']:.2f} (raw {raw_acc['strip']:.2f})  label_mass={mass:.1%}")
    for r in diag.itertuples():
        print(f"     [{LABELS[r.label]}] {r.sentence[:50]!r} → {r.answer!r}")

## 6. Model-level analysis

- **drop** = clean accuracy − noisy accuracy, with a 95% paired bootstrap CI (resampling items).
- **norm_drop** = drop ÷ (clean accuracy − 1/3): the share of the model's above-chance ability that the noise removes. 1.0 = back to chance. This is fair across models that start at different levels.
- **ppl_ratio** = exp(NLL per syllable on noisy − on clean).
- Models below `MIN_CLEAN_ACC` on clean text are flagged and left out of the correlations: they cannot drop, so they would fake a pattern (that happened in v1).

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore", message="An input array is constant")

done = [m for m in MODELS if os.path.exists(pred_path(m))]
preds = pd.concat([pd.read_csv(pred_path(m)) for m in done])
preds["correct"] = (preds.pred == preds.label).astype(int)
preds["correct_raw"] = (preds.pred_raw == preds.label).astype(int)

rng = np.random.default_rng(SEED)
rows = []
for m, gm in preds.groupby("model"):
    clean = gm[gm.cond == "clean"].set_index("item_id")
    nll_clean = clean.nll.sum() / clean.clean_syl.sum()
    for c in CONDITIONS:
        g = gm[gm.cond == c].set_index("item_id").loc[clean.index]
        diff = (clean.correct - g.correct).values
        boot = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(1000)]
        rows.append({
            "model": m, "cond": c,
            "acc": g.correct.mean(), "acc_raw": g.correct_raw.mean(),
            "macro_f1": f1_score(g.label, g.pred, average="macro"),
            "label_mass": g.label_mass.median(),
            "drop": diff.mean(), "drop_lo": np.percentile(boot, 2.5), "drop_hi": np.percentile(boot, 97.5),
            "ppl_ratio": np.exp(g.nll.sum() / g.clean_syl.sum() - nll_clean),
        })
summary = pd.DataFrame(rows).merge(tok_stats, on=["model", "cond"])
clean_rows = summary[summary.cond == "clean"].set_index("model")
above_chance = summary.model.map(clean_rows.acc) - 1 / 3
summary["norm_drop"] = (summary["drop"] / above_chance).where(above_chance > 0.05)  # undefined near chance
for k in ["fertility", "broken_rate", "fragment_rate"]:
    summary[f"d_{k}"] = summary[k] - summary.model.map(clean_rows[k])
summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)

OK_MODELS = clean_rows.index[clean_rows.acc >= MIN_CLEAN_ACC].tolist()
print("In the analysis:", OK_MODELS)
print("Left out (clean accuracy below gate):", sorted(set(done) - set(OK_MODELS)))
if failed:
    print("Failed to run:", failed)

summary.pivot(index="model", columns="cond", values="acc")[CONDITIONS].round(3)

In [ ]:
# raw vs calibrated on clean text, and how much probability the model puts on the labels at all
clean_rows[["acc_raw", "acc", "macro_f1", "label_mass"]].round(3)

In [ ]:
summary[summary.model.isin(OK_MODELS)].pivot(index="model", columns="cond", values="norm_drop")[CONDITIONS[1:]].round(2)

In [ ]:
noisy = summary[(summary.cond != "clean") & summary.model.isin(OK_MODELS)]
corr = pd.DataFrame([
    {"predictor": k, "spearman_rho": spearmanr(noisy[k], noisy["norm_drop"])[0], "p": spearmanr(noisy[k], noisy["norm_drop"])[1]}
    for k in ["d_fertility", "d_broken_rate", "d_fragment_rate", "ppl_ratio"]
])
print(f"{len(noisy)} (model × noisy condition) points from {len(OK_MODELS)} models; target = norm_drop")
corr.round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for m, g in summary.groupby("model"):
    g = g.set_index("cond").loc[CONDITIONS]
    axes[0].plot(CONDITIONS, g.acc, marker="o", ls="-" if m in OK_MODELS else ":", label=m.split("/")[1])
axes[0].axhline(1 / 3, color="gray", ls=":", lw=1)
axes[0].set(title="Accuracy by condition (dotted line = chance; dotted model = below gate)", ylabel="accuracy")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(fontsize=7)
for ax, k in zip(axes[1:], ["d_fertility", "ppl_ratio"]):
    for m, g in noisy.groupby("model"):
        ax.scatter(g[k], g["norm_drop"], label=m.split("/")[1])
    ax.set(title=f"normalised drop vs {k}", xlabel=k, ylabel="norm_drop")
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/pilot_v2_plots.png", dpi=150)
plt.show()

## 7. Item-level analysis: tokenizer or exposure?

Take every (model, item, noisy condition) where the model was **right on the clean version**. Did it **flip** to wrong?

Predictors, all standardised (z-scores), per item:
- `d_tok`: extra tokens per syllable caused by the noise — **tokenizer**.
- `broken`: share of the noisy sentence's syllables split into several tokens — **tokenizer**.
- `d_nll`: extra NLL per syllable on the noisy sentence — **exposure** (how unfamiliar the noisy text is to the model).

Three logistic regressions, each with model and condition fixed effects (so the noise level and the model's overall skill are held constant), with standard errors clustered by item. Compare **AIC** (lower = better fit) and the coefficients.

⚠️ `d_tok` and `d_nll` are correlated (more tokens usually means more NLL); the correlation is printed first. If it is very high, the "both" model cannot separate them and you need a design that does (e.g. the same-tokenizer models).

In [ ]:
import statsmodels.formula.api as smf

it = preds[preds.model.isin(OK_MODELS)].merge(tok_items, on=["model", "item_id", "cond"], suffixes=("", "_tok"))
it["nll_syl"] = it.nll / it.clean_syl
base = it[it.cond == "clean"][["model", "item_id", "correct", "n_tok", "nll_syl"]]
x = it[it.cond != "clean"].merge(base, on=["model", "item_id"], suffixes=("", "_clean"))
x = x[(x.correct_clean == 1) & (x.n_syl > 0)].copy()
x["flip"] = 1 - x.correct
x["d_tok"] = (x.n_tok - x.n_tok_clean) / x.clean_syl
x["broken"] = x.n_broken / x.n_syl
x["d_nll"] = x.nll_syl - x.nll_syl_clean
for k in ["d_tok", "broken", "d_nll"]:
    x[f"z_{k}"] = (x[k] - x[k].mean()) / x[k].std()
x.to_csv(f"{OUT_DIR}/item_level.csv", index=False)

print(f"{len(x)} rows, flip rate {x.flip.mean():.2f}, models: {x.model.nunique()}")
print("correlation between predictors:\n", x[["d_tok", "broken", "d_nll"]].corr().round(2))

fixed = "C(model) + C(cond)"
fits = {}
for label, f in {
    "tokenizer": f"flip ~ z_d_tok + z_broken + {fixed}",
    "exposure": f"flip ~ z_d_nll + {fixed}",
    "both": f"flip ~ z_d_tok + z_broken + z_d_nll + {fixed}",
}.items():
    fits[label] = smf.logit(f, data=x).fit(disp=0, cov_type="cluster", cov_kwds={"groups": x.item_id})

compare = pd.DataFrame({k: {"AIC": v.aic, "pseudo_R2": v.prsquared} for k, v in fits.items()}).T
print("\nmodel comparison (lower AIC = better):\n", compare.round(3))
b = fits["both"]
coef = pd.DataFrame({"coef": b.params, "lo": b.conf_int()[0], "hi": b.conf_int()[1], "p": b.pvalues})
coef.loc[["z_d_tok", "z_broken", "z_d_nll"]].round(3)

## 8. What to write in your paper note

1. **Diagnostics first.** For any model below the gate: is `label_mass` tiny, or do its greedy answers look like a different format? If so, the prompt (not the model) is the problem. Fix that before drawing conclusions.
2. **Same tokenizer, different data.** Compare Qwen2.5-7B, Qwen3-8B and SeaLLMs-v3-7B on `norm_drop` for `strip`. Their tokenizer statistics are identical. Any difference is exposure, not tokenization.
3. **Item level.** Which fits better, `tokenizer` or `exposure` (AIC)? In `both`, which coefficient survives? That is your answer to the 1A question, in a form reviewers can check.
4. **Mixed vs teen code vs strip.** Is `mixed` worse than either part alone (do the effects add up)?
5. Look at 20 flipped items (below). What kinds of words broke?

Then: `Runtime → Disconnect and delete runtime`.

In [ ]:
m = OK_MODELS[0] if OK_MODELS else done[0]
p = preds[preds.model == m].pivot(index="item_id", columns="cond", values="correct")
flipped = p[(p.clean == 1) & (p.strip == 0)].index[:20]
data[(data.item_id.isin(flipped)) & (data.cond.isin(["clean", "strip"]))].pivot(index="item_id", columns="cond", values="text")